# 14. How much each protection matters (the leakage ladder)

The ridge models of step 7 (TF-IDF, Qwen3-Embedding-8B, Mistral-Small-24B) under weaker protocols, one protection
added at a time, always tested on the **same** 25,524 articles of the 226 outlets of the paper, in the same folds:

| rung | training outlets and articles | text |
|---|---|---|
| article split | the 226 outlets, folds of articles (an outlet is on both sides) | anonymous |
| L0 | every outlet with ≥ 10 articles at "published from 2025": sisters and copies included | original |
| L1 | + duplicates removed across outlets: sisters still included | original |
| L2 | + one outlet per ownership group (the 226) | original |
| L3 | the 226 | + links, e-mails, the outlet's own name |
| L4 | the 226 | + GLiNER (the paper, step 7) |

Sisters and copies come back from the step-2 cache with the same functions and constants (checked against the paper's
articles); details in `ladder.py`. Order of the work: the stages, TF-IDF on every rung, the frozen models on the article
split (vectors of step 5), then the new vectors (rules text: 226 outlets; original text: all L0 articles) on the GPU and
the frozen models on L0–L3.

In [ ]:
N_JOBS = 16
GPU_GB = 19
QWEN_BUDGET = 16384
MISTRAL_BUDGET = 8192

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import ladder as LA
import paper as PA

A, by_step = LA.stages(N_JOBS, LOG)
print(f"L0: {A.domain.nunique()} outlets, {len(A):,} articles | L1: {A[A.s1].domain.nunique()} outlets, "
      f"{A.s1.sum():,} | L2-L4: {A[A.s2].domain.nunique()} outlets, {A.s2.sum():,}")
by_step["last step"].value_counts()

## TF-IDF on every rung, the frozen models on the article split

In [ ]:
for rung in LA.RUNGS:
    LA.run(rung, A, ["TF-IDF"], N_JOBS, LOG)
LA.run("article split", A, ["Qwen3-8B", "Mistral-24B"], N_JOBS, LOG)

## New frozen vectors (GPU) and the frozen models on L0–L3

In [ ]:
LA.embed("rules", A, LOG, GPU_GB, QWEN_BUDGET, MISTRAL_BUDGET)
LA.run("L3", A, ["Qwen3-8B", "Mistral-24B"], N_JOBS, LOG)
LA.embed("original", A, LOG, GPU_GB, QWEN_BUDGET, MISTRAL_BUDGET)
for rung in ["L0", "L1", "L2"]:
    LA.run(rung, A, ["Qwen3-8B", "Mistral-24B"], N_JOBS, LOG)

## Results (the same outlets and test articles for every rung)

In [ ]:
art, outlets, rows = PA.load()
L = PA.ladder_results(outlets)
L.to_csv(C.OUT / "paper_ladder.csv", index=False)
L.assign(text=[f"{v:.3f}" if m == "Spearman" else f"{v:.1f}" for v, m in zip(L.value, L.measure)]) \
    .pivot_table(index="rung", columns=["measure", "model"], values="text", aggfunc="first").reindex(LA.RUNGS)